# Customer 360 Accelerator
### Notebook 01 — Create Semantic Model

---

> **Purpose:** Trigger creation of the default Semantic Model from the Lakehouse
> Delta table, then verify it exists. The Semantic Model provides the schema
> layer that the Ontology and Data Agent will reference.
>
> **Prerequisite:** Run `00_setup_and_load_data` first.

---
## Configuration

In [ ]:
# ── Paste outputs from Notebook 00, or leave blank to auto-detect ────────────
WORKSPACE_ID   = ""
WORKSPACE_NAME = ""
LAKEHOUSE_ID   = ""
LAKEHOUSE_NAME = "Customer360Lakehouse"
TABLE_NAME     = "Customer360"

In [ ]:
import sempy.fabric as fabric
import time

client = fabric.FabricRestClient()

# Auto-detect workspace
if not WORKSPACE_ID:
    WORKSPACE_ID = fabric.get_notebook_workspace_id()
if not WORKSPACE_NAME:
    WORKSPACE_NAME = fabric.resolve_workspace_name()

# Auto-detect lakehouse ID
if not LAKEHOUSE_ID:
    resp = client.get(f"v1/workspaces/{WORKSPACE_ID}/lakehouses")
    resp.raise_for_status()
    lh = next(
        (l for l in resp.json().get("value", []) if l["displayName"] == LAKEHOUSE_NAME),
        None,
    )
    if not lh:
        raise ValueError(f"Lakehouse '{LAKEHOUSE_NAME}' not found. Run Notebook 00 first.")
    LAKEHOUSE_ID = lh["id"]

print(f"Workspace : {WORKSPACE_NAME} ({WORKSPACE_ID})")
print(f"Lakehouse : {LAKEHOUSE_NAME} ({LAKEHOUSE_ID})")

---
## Step 1 — Check for Existing Semantic Model

Fabric auto-creates a default Semantic Model when a Lakehouse has tables.
The default model shares the same `displayName` as the Lakehouse.

In [ ]:
def find_semantic_model(workspace_id, lakehouse_name):
    """Search for a Semantic Model matching the Lakehouse name."""
    resp = client.get(f"v1/workspaces/{workspace_id}/items?type=SemanticModel")
    resp.raise_for_status()
    for item in resp.json().get("value", []):
        if item.get("displayName") == lakehouse_name:
            return item["id"]
    return None

SEMANTIC_MODEL_ID = find_semantic_model(WORKSPACE_ID, LAKEHOUSE_NAME)

if SEMANTIC_MODEL_ID:
    print(f"Semantic Model already exists: {LAKEHOUSE_NAME} (ID: {SEMANTIC_MODEL_ID})")
else:
    print(f"Semantic Model not found yet. Will trigger creation in the next step.")

---
## Step 2 — Trigger Default Semantic Model Creation

If the Semantic Model doesn't exist, we trigger its creation via the
Lakehouse API. This tells Fabric to auto-generate a Direct Lake
semantic model from the Delta tables in the Lakehouse.

In [ ]:
if not SEMANTIC_MODEL_ID:
    print("Triggering default Semantic Model creation...")
    
    # Method 1: Use the triggerTableDefaultSemanticModel endpoint
    trigger_resp = client.post(
        f"v1/workspaces/{WORKSPACE_ID}/lakehouses/{LAKEHOUSE_ID}/triggerTableDefaultSemanticModel"
    )
    
    if trigger_resp.status_code in (200, 201, 202):
        print(f"   Trigger accepted (HTTP {trigger_resp.status_code}).")
        
        # Handle async (202) case
        if trigger_resp.status_code == 202:
            op_id = trigger_resp.headers.get("x-ms-operation-id")
            retry_after = int(trigger_resp.headers.get("Retry-After", "30"))
            if op_id:
                print(f"   Polling operation {op_id}...")
                while True:
                    time.sleep(retry_after)
                    poll = client.get(f"v1/operations/{op_id}")
                    poll.raise_for_status()
                    status = poll.json().get("status")
                    print(f"   Status: {status}")
                    if status == "Succeeded":
                        break
                    elif status in ("Failed", "Cancelled"):
                        print(f"   Semantic Model trigger {status}.")
                        break
    else:
        print(f"   Trigger returned HTTP {trigger_resp.status_code}: {trigger_resp.text[:200]}")
        print("   This may be expected if the API version doesn't support this endpoint.")
        print("   Fabric often auto-creates the semantic model — polling below will check.")
else:
    print("Semantic Model already exists — skipping trigger.")

---
## Step 3 — Poll for Semantic Model

The Semantic Model may take 30–120 seconds to materialise after the trigger.
We poll every 15 seconds until it appears.

In [ ]:
MAX_POLLS = 12   # 12 x 15s = 3 minutes max
POLL_WAIT = 15   # seconds

if not SEMANTIC_MODEL_ID:
    print("Waiting for Semantic Model to appear...")
    for attempt in range(1, MAX_POLLS + 1):
        SEMANTIC_MODEL_ID = find_semantic_model(WORKSPACE_ID, LAKEHOUSE_NAME)
        if SEMANTIC_MODEL_ID:
            print(f"\nSemantic Model found: {LAKEHOUSE_NAME} (ID: {SEMANTIC_MODEL_ID})")
            break
        
        # Also check Lakehouse properties for the SM ID
        lh_resp = client.get(f"v1/workspaces/{WORKSPACE_ID}/lakehouses/{LAKEHOUSE_ID}")
        if lh_resp.status_code == 200:
            props = lh_resp.json().get("properties", {})
            sm_id_from_props = props.get("defaultDatasetId") or props.get("semanticModelId")
            if sm_id_from_props:
                SEMANTIC_MODEL_ID = sm_id_from_props
                print(f"\nSemantic Model found via Lakehouse properties (ID: {SEMANTIC_MODEL_ID})")
                break
        
        print(f"   [{attempt}/{MAX_POLLS}] Not ready yet — waiting {POLL_WAIT}s...")
        time.sleep(POLL_WAIT)
    
    if not SEMANTIC_MODEL_ID:
        print("\nSemantic Model did not appear within the polling window.")
        print("This usually means the Fabric capacity is paused or the table is empty.")
        print("Check the Fabric portal manually.")

print(f"\nSEMANTIC_MODEL_ID = \"{SEMANTIC_MODEL_ID}\"")

---
## Verification

List all Semantic Models in the workspace to confirm.

In [ ]:
resp = client.get(f"v1/workspaces/{WORKSPACE_ID}/items?type=SemanticModel")
resp.raise_for_status()
models = resp.json().get("value", [])

print(f"Semantic Models in workspace ({len(models)} total):")
for m in models:
    marker = " <-- THIS ONE" if m["id"] == SEMANTIC_MODEL_ID else ""
    print(f"   {m['displayName']} (ID: {m['id']}){marker}")

In [ ]:
# Store IDs for downstream notebooks
print("\n" + "=" * 60)
print("OUTPUTS — Copy these to the next notebook if needed")
print("=" * 60)
print(f"WORKSPACE_ID      = \"{WORKSPACE_ID}\"")
print(f"WORKSPACE_NAME    = \"{WORKSPACE_NAME}\"")
print(f"LAKEHOUSE_ID      = \"{LAKEHOUSE_ID}\"")
print(f"LAKEHOUSE_NAME    = \"{LAKEHOUSE_NAME}\"")
print(f"TABLE_NAME        = \"{TABLE_NAME}\"")
print(f"SEMANTIC_MODEL_ID = \"{SEMANTIC_MODEL_ID}\"")